**Primeira Versão Publicada em:** 11 de Julho de 2026  
**Última atualização deste notebook em:** 24 de Julho de 2026 

# **Aula 00 - Introdução ao Curso de Galerkin Descontínuo (DG) 1D**

### **Da Literatura à Implementação em Python**

Bem-vindos ao nosso curso prático! O objetivo da nossa jornada é claro: 

> Construir um *solver* computacional "do zero" em aproximadamente 14 aulas. 

Em cada aula, antes de escrevermos cada linha de código, vamos entender as razões teóricas e práticas por trás desta metodologia.

A pergunta motivacional é: 

### **Por que utilizar o método Galerkin Descontínuo (DG)?**

Uma resposta simples e que frequentemente é dada na literatura é que podemos descrever o DG como: *uma união inteligente do Método dos Elementos Finitos (MEF) com o Método dos Volumes Finitos (VF), combinando as melhores características e formulações de ambos*. 

Suas principais vantagens incluem:
* **Elementos independentes:** A formulação garante que a aproximação e os cálculos da solução sejam realizados de forma isolada dentro de cada célula (elemento).
* **Flexibilidade espacial:** O método possui uma capacidade natural para lidar com geometrias complexas e malhas não estruturadas.
* **Implementação de contornos:** Graças à comunicação estabelecida pelas fronteiras via teoria de volumes finitos, a fácil implementação de condições de contorno é garantida.
* **Convergência Exponencial:** Um dos atrativos do DG aplicado a elementos de alta ordem é a precisão atingida. Quando avaliamos a simulação de problemas clássicos (como a equação de Burgers), podemos refinar o modelo de duas maneiras diferentes para reduzir o erro da solução numérica, conforme evidenciado pelos gráficos abaixo:

<div align="center">
  <img src="https://raw.githubusercontent.com/properallan/CFD4SciML/main/DG/Lessons/images/aula00_conv.png" alt = "Curvas de convergência para os casos de refinamento de p e h fixo e, refinamento de h e p fixo." width="700"><br>
  <em>Curvas de convergência para os casos de refinamento de p e h fixo e, refinamento de h e p fixo. Figura retirada de Moura (2011)</em>
</div>

* **Refinamento $h$ (Gráfico Log-Log):** Quando mantemos a ordem do polinômio de aproximação fixa e optamos por diminuir o tamanho dos elementos (aumentando a quantidade de partições no domínio), a taxa do erro apresenta uma **convergência algébrica**.
* **Refinamento $p$ (Gráfico Semi-Log):** Quando mantemos a malha estática e aumentamos o grau $P$ do polinômio aproximador dentro do próprio elemento, o erro colapsa com uma **convergência exponencial**. Isso nos permite simular soluções extremamente precisas sem precisar explodir o número de graus de liberdade.

> Para um mesmo nível de precisão, métodos de alta ordem frequentemente necessitam de muito menos graus de liberdade do que métodos tradicionais de baixa ordem, reduzindo custo computacional em diversas aplicações.

### **Quando o DG vale a pena?**

Dentro do universo de métodos numéricos o DG vale a pena quando queremos resolver problemas que contenham ondas de choque, que necessitam de malhas não estruturas, que precisam de alto refinamento (podendo utilizar alta ordem polinomial), em resumo a recomendação de métodos é dada pela tabela

<div align="center">

| Método             | Pontos fortes                              |
| :------------------ | :------------------------------------------ |
| Diferenças Finitas | simples e eficiente em malhas estruturadas |
| Volumes Finitos    | conservação e robustez                     |
| Elementos Finitos  | geometrias complexas                       |
| DG                 | alta ordem + conservação + paralelização   |

</div>

---

## **1. O que veremos e construiremos ao longo do curso**

O método Galerkin Descontínuo é composto por diversas estratégias matemáticas e cada uma delas se conversa de forma sequencial, a figura abaixo exemplifica um "mapa mental" de cada teoria necessária para construir o nosso *solver*. 

<div align="center">
  <img src="https://raw.githubusercontent.com/properallan/CFD4SciML/main/DG/Lessons/images/aula00_solver.png" alt="Blocos teóricos para o desenvolvimento do solver DG" width="800"><br>
  <em>Blocos teóricos para o desenvolvimento do solver DG</em>
</div>

Nas seções seguintes, cada um desses tópicos será brevemente apresentado para que o leitor vá se familiarizando com os diferentes assuntos.

## **2. Base Matemática: O Espaço Espectral**

O nosso objetivo principal ao decorrer das aulas é compreender a teoria matemática para implementar as expansões espaciais representadas na Equação 1 e, por consequência, na Equação 2.

A solução numérica local dentro do domínio de cada elemento é definida pela série:
$$
u_{e}^{\delta}(x)=\sum_{i=0}^{p}c_{i}^{(e)}\phi_{i}(\xi_{e}(x)) \quad \text{para} \quad \xi_{e}\in[-1,1] \tag{1}
$$

Considerando o comportamento evolutivo da formulação:
$$
u_{e}^{\delta}(x,t)=\sum_{i=0}^{P}c_{i}^{(e)}(t)\phi_{i}(\xi_{e}(x)) \quad \text{para} \quad x\in[x_{e}^{-},x_{e}^{+}] \tag{2}
$$

Para construir o espaço espectral $\phi_i$, a **Aula 01** será dedicada a desenvolver códigos que lidem com as funções geradoras e regras de recorrência dos polinômios ortogonais. A recorrência fundamental obedece a relação:
$$a_{k}^{(1)}\mathcal{P}_{k+1}^{\alpha,\beta}(\xi)=a_{k}^{(2)}\mathcal{P}_{k}^{\alpha,\beta}(\xi)-a_{k}^{(3)}\mathcal{P}_{k-1}^{\alpha,\beta}(\xi)$$

Onde os pesos polinomiais que controlam a família de Jacobi são computados através de:
$$a_{k}^{(1)}=2(k+1)(k+\alpha+\beta+1)(2k+\alpha+\beta)$$
$$a_{k}^{(2)}=(2k+\alpha+\beta+1)(\alpha^{2}-\beta^{2})+\xi(2k+\alpha+\beta+2)!/(2k+\alpha+\beta-1)!$$
$$a_{k}^{(3)}=2(k+\alpha)(k+\beta)(2k+\alpha+\beta+2)$$

Para derivadas dos Polinômios Ortogonais, será garantida a relação:
$$\frac{d^{m}}{dx^{m}}\mathcal{P}_{k}^{\alpha,\beta}(\xi)=2^{-m}\frac{(k+\alpha+\beta+m)!}{(k+\alpha+\beta)!}\mathcal{P}_{k-m}^{\alpha+m,\beta+m}(\xi)$$

---

## **3. Integração Numérica e a Forma Fraca**

Para traduzir a teoria em computação matricial rápida, nós não resolvemos integrais de forma analítica nas entranhas do solver. Todo o processo pode ser modularizado da seguinte maneira:
1. A integral exata é traduzida e resolvida numericamente no código através da obtenção e ponderação de nós e **pesos e raízes** de quadratura (temas das **Aulas 02 e 03**).

$$\int_{\Omega_{pd}} \phi_i \phi_j d \xi \approx \sum w_i P(x_i)$$

2. A montagem das **bases Ortogonais** vinculadas e aplicadas aos resultados dessas integrações para a interpolação exata serão detalhadas.

Os textos de referência do nosso curso (ver eq.33 de Moura (2011)) condensam grande parte desses operadores algébricos diretamente na Equação 3. 

$$
\underbrace{J_e \left[\int_{\Omega_{pd}} \phi_i \phi_j d \xi \right]}_{\text{Jacobiano e Matriz de Massa} \ \mathcal{M}} 
\dfrac{\partial}{\partial t} \begin{Bmatrix} c_0 \\ c_1 \\ \vdots \\ c_P \end{Bmatrix} = 
\underbrace{\int_{\Omega_{pd}} f \dfrac{\partial}{\partial \xi} \begin{Bmatrix} \phi_0 \\ \phi_1 \\ \vdots \\ \phi_P \end{Bmatrix} d \xi }_{\text{Fluxo, Matriz de Derivação} \ \mathcal{D} \ \text{e Rigidez} \ \mathcal{S}} - 
\underbrace{\tilde{f}_{e,e+1} \begin{Bmatrix} \phi^+_0 \\ \phi^+_1 \\ \vdots \\ \phi^+_P \end{Bmatrix} + 
\tilde{f}_{e-1,e}\begin{Bmatrix} \phi^-_0 \\ \phi^-_1 \\ \vdots \\ \phi^-_P \end{Bmatrix}}_{\text{Fluxos nas Fronteiras}} \tag{3}
$$

A partir dessa equação global, o nosso curso irá construir tijolo por tijolo cada operador:
* Formulação da **Matriz de Derivação via Vandemonde** na **Aula 04** e a **Matriz de Derivação via Colocação** na **Aula 05** 
* Na **Aula 06** vamos construir o *core* principal dessa formulação: **Geometria, Jacobiano** as Matrizes de **Massa, Rigidez, Lift**.
* Investigaremos o fluxo próximo a matriz de rigidez com a estratégia de **Projeção de Fluxos** na **Aula 07** 
* Já na **Aula 08** iremos investigar **Fluxos Numéricos** que melhoram a interação nas *fronteiras* dos elementos.
* Iremos configurar as **Condições Iniciais e de Contorno** na **Aula 09**.
* Para que na **Aula 10** haja a estruturação e fechamento do **Operador Espacial Discreto** governante $L_h$ (vide equação 4), que configura passar o Jacobiano e Matriz de Massa para o lado direito para que assim possa ser feita a marcha no tempo.

$$
L_h = \dfrac{\partial}{\partial t} \begin{Bmatrix} c_0 \\ c_1 \\ \vdots \\ c_P \end{Bmatrix} = \frac{1}{J_e} \left[\int_{\Omega_{pd}} \phi_i \phi_j d \xi \right]^{-1} \left (
\int_{\Omega_{pd}} f \dfrac{\partial}{\partial \xi} \begin{Bmatrix} \phi_0 \\ \phi_1 \\ \vdots \\ \phi_P \end{Bmatrix} d \xi- 
\tilde{f}_{e,e+1} \begin{Bmatrix} \phi^+_0 \\ \phi^+_1 \\ \vdots \\ \phi^+_P \end{Bmatrix} + 
\tilde{f}_{e-1,e}\begin{Bmatrix} \phi^-_0 \\ \phi^-_1 \\ \vdots \\ \phi^-_P \end{Bmatrix} \right ) \tag{4}
$$

---

## **4. Comunicação entre Elementos: O Fluxo Numérico**

Como a descontinuidade é tolerada nas fronteiras espaciais, a transferência de momento e massa entre células ocorre através da aproximação em interface via fluxo. O universo de capturadores de onda no CFD é vasto e existem diversos tipos de fluxos, normalmente separados em **Convectivos** e **Difusivos**. Além de transmitir informação entre elementos, os fluxos são capazes de impor estabilidade e selecionar a solução física quando existem múltiplas soluções fracas.

Um exemplo de fluxo numérico comum na literatura é o **Fluxo de Lax-Friedrichs** (também conhecido como fluxo de Rusanov).

Para formulações em que a conservação é dada por um campo Escalar o fluxo de Lax-Friedrichs é descrito por:
$$\tilde{f}(u_{E},u_{D})=\frac{1}{2}(f(u_{E})+f(u_{D}))-\frac{|\tilde{\lambda}|}{2}(u_{D}-u_{E})$$

Nas bordas, os limites Direitos e Esquerdos são calculados diretamente pelas suas projeções polinomiais sobre a fronteira:
$$u_{D}=u_{e+1}^{\delta}(x_{e+1}^{-})=\sum_{i=0}^{P}c_{i}^{(e+1)}\phi_{i}(-1) \hspace{2cm} u_{E}=u_{e}^{\delta}(x_{e}^{+})=\sum_{i=0}^{P}c_{i}^{(e)}\phi_{i}(+1)$$

Para modelos governados por um Sistema acoplado e hiperbólico ele se torna:
$$\tilde{\vec{F}}(\vec{U}_{E},\vec{U}_{D})=\frac{1}{2}(\vec{F}(\vec{U}_{E})+\vec{F}(\vec{U}_{D}))-\frac{\lambda_{max}}{2}(\vec{U}_{D}-\vec{U}_{E})$$

Na **Aula 08** dedicaremos nosso foco na investigação dessa e de outras estratégias (como as condições **Upwind, fluxos de Roe, HLL, HLLC** e o esquema de **Bassi-Rebay**).

---

## **5. Padronizando a Malha: O Jacobiano**

O grande trunfo dos métodos nodais em malhas é a independência dos algoritmos quanto às geometrias deformadas do mundo físico. Como veremos na **Aula 06**, nós processamos a lógica matemática padronizando as distâncias dos elementos.

Mapeamos a coordenada global $x\in[x_{e}^{-},x_{e}^{+}]$ para um domínio local restrito da forma $\xi\in[-1,1]$ aplicando as seguintes transformações lineares no espaço 1D:

<div align="center">
  <img src="https://raw.githubusercontent.com/properallan/CFD4SciML/main/DG/Lessons/images/aula00_jacobiano.png" alt="Mapeamento de coordenadas" width="800"><br>
  <em>Mapeamento de coordenadas</em>
</div>

sendo o tamanho do elemento definido por $h_e = x_e^+ - x_e^-$ temos que

* Da coordenada global para o elemento padrão:
$$\xi_{e}(x)=2\frac{x-x_{e}^{-}}{h_e}-1$$

* Do elemento padrão isolado de volta à dimensão real:
$$x_{e}(\xi)=x_e^- + \frac{1+\xi}{2}h_e$$

---

## **6. Marcha no Tempo**

Assim que o operador computacional espacial estiver completamente modelado, a equação governante se torna um modelo reduzido semi-discreto dependente unicamente do tempo:
$$\frac{\partial U}{\partial t}=L_{h}(U)$$

O acoplamento e evolução desse sistema será conduzido em etapas usando integradores explícitos conhecidos como esquemas *Runge-Kutta Total Variation Diminishing* (**RKTVD**) ou ainda *Runge-Kutta Strong Stability Preserving* (**RKSSP**). A exemplo os RKSSP de 2ª e 3ª ordem:

Para 2ª ordem no tempo:
$$\begin{align*}
\{c_{i}\}^{*}=& \ \{c_{i}\}^{(t)}+\Delta t\partial_{t}\{c_{i}\}^{(t)} \\
\{c_{i}\}^{(t+\Delta t)}=& \ (\{c_{i}\}^{(t)}+\{c_{i}\}^{*}+\Delta t\partial_{t}\{c_{i}\}^{*})/2
\end{align*}
$$

Para 3ª ordem no tempo:
$$\begin{align*} 
\{c_{i}\}^{*}= & \ \{c_{i}\}^{(t)}+\Delta t\partial_{t}\{c_{i}\}^{(t)} \\
\{c_{i}\}^{**}= & \ (3\{c_{i}\}^{(t)}+\{c_{i}\}^{*}+\Delta t\partial_{t}\{c_{i}\}^{*})/4 \\
\{c_{i}\}^{(t+\Delta t)}= & \ (\{c_{i}\}^{(t)}+2\{c_{i}\}^{**}+2\Delta t\partial_{t}\{c_{i}\}^{**})/3
\end{align*}
$$

A formulação desses e de outros integradores de ordem mais elevada será desenvolvida e coberta nas **Aulas 13 e 14**. Como a marcha no tempo usa integradores explícitos (RKSSP), o passo de tempo $\Delta t$ é rigorosamente limitado pela Condição de **Courant-Friedrichs-Lewy** (CFL). No DG, essa restrição escala com a ordem do polinômio aproximador (geralmente $\approx 1/(2P+1)$).

Além de integrar o tempo, como polinômios de grau excessivo podem gerar espúrios não-físicos na presença de choques acentuados, as formulações numéricas devem obrigatoriamente sofrer correções espaciais via **limitadores**. Abordaremos estratégias conhecidas, como:
* minmod,
* superbee,
* Barth-Jespersen,
* van Leer,

entre outras, que fazem parte do grupo de **Slope Limiters** reservando este tema para as **Aulas 11 e 12**.

---

## **7. Quais problemas poderemos resolver assim que finalizarmos o solver?**

Com todo esse escopo computacional e a nossa biblioteca base pronta, encerraremos nossa jornada com um *solver* versátil o suficiente para resolver problemas e fenômenos não lineares aplicados a problemas em uma dimensão. O horizonte de equações que o nosso modelo irá conseguir solucionar abriga problemas clássicos do CFD, como:
* Escoamento e Advecção linear
* Equação de Burgers
* A variação do modelo de Burgers Não-viscoso
* O modelo macroscópico e acoplado de Tráfego (*Traffic Model*)

entre outros, que serão disponibilizados com o encerramento do curso.

---

> Ao término do curso, teremos desenvolvido uma implementação própria do método de Galerkin Descontínuo para problemas unidimensionais. Mais importante do que obter um código funcional será compreender cada operador matemático presente no algoritmo, desde a construção da base espectral até a integração temporal. Esse entendimento permitirá estender naturalmente o solver para sistemas de equações, problemas bidimensionais e aplicações mais avançadas em CFD.

---

## **8. Cronograma de Aulas**
**Observações:** 
> * O cronograma poderá sofrer alterações ao decorrer da construção das aulas e notebooks
> * Os links de cada aula serão adicionados somente ao fim do curso
> * As funções geradas em cada aula serão adicionados somente ao fim do curso

<div align="center">

|**Aula** | **Tópico** | **Funções** |
|:-------:|:-----:|:-----:|
|00| Introdução Curso de Galerkin Descontínuo 1D||
|01| Polinômios de Jacobi ||
|02| Quadraturas||
|03| Quadraturas via Polinômios de Jacobi||
|04| Matriz de Diferenciação via Vandermonde||
|05| Matriz de Diferenciação via Colocação||
|06| Fundamentos Geométricos e Matrizes Clássicas do DG 1D||
|07| Projeção de Fluxos||
|08| Fluxos Numéricos||
|09| Condições Iniciais e de Contorno||
|10| Construção do Operador Lh||
|11| Slope Limiters e o Fenômeno de Gibbs - Parte 1||
|12| Slope Limiters e o Fenômeno de Gibbs - Parte 2||
|13| Método de Runge-Kutta Strong Stability Preserving (RKSSP) - Parte 1||
|14| Método de Runge-Kutta Strong Stability Preserving (RKSSP) - Parte 2||
|15| Resolução de Problemas com *solver* DG1D (Advecção, Burgers, Traffic Model etc) | |

</div>



## **9. Bibliografias**
### **9.1 Principal**

A teoria apresentada, assim como o código a ser desenvolvido, se pauta principalmente nas três referências a seguir: 
*   **G. E. Karniadakis & S. J. Sherwin:** *Spectral/hp Element Methods for Computational Fluid Dynamics*.
*   **J. S. Hesthaven & T. Warburton:** *Nodal Discontinuous Galerkin Methods: Algorithms, Analysis, and Applications*.
*   **J. Moura, et al.:** *Uma Introdução Prática ao Método Galerkin Descontínuo para Aplicações em Dinâmica dos Fluidos*.

### **9.2 Complementares**
> Serão adicionadas posteriormente